## Generators in Python – Theory Notes
**1. What Are Generators?**

**Definition:**
Generators are special functions in Python that return values one at a time using the yield keyword instead of returning everything at once.

**Why Are Generators Useful?**

- Save memory (do not store entire data in RAM)

- Faster for large datasets

- Produce values only when needed (lazy evaluation)

- Ideal for streaming or continuous data

**2. How Generators Work**

Generators pause and resume execution using yield.
```
Example:

def numbers():
    yield 1
    yield 2
    yield 3

```
Calling this function does not run it immediately.
It returns a generator object that produces values one by one.

***3. Lazy Data Loading***

**Definition:**
Lazy loading means data is generated only when required, not all at once.

Example of Lazy Loading
def read_lines(file):
    for line in open(file):
        yield line


**Benefits:**

- Handles large files easily

- Efficient memory usage

- Useful for pipelines, streaming, APIs

**4. Generator Expression**

A short syntax similar to list comprehension but lazy:

squares = (x*x for x in range(10))


This does NOT create a full list; values are generated when needed.

## Coding Tasks

**1. Create a fetch_page(url) generator that sends a GET request and yields the raw HTML of each page until no next page exists.**

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def fetch_page(url):
    """Generator that yields HTML of each page until no next page exists."""
    
    while url:
        response = requests.get(url)
        html = response.text
        yield html
        
        soup = BeautifulSoup(html, "html.parser")
        
        next_link = soup.find("li", class_="next")
        
        if not next_link or not next_link.a:
            break
        url = urljoin(url, next_link.a["href"])

# start from page 1
gen = fetch_page("http://books.toscrape.com/catalogue/page-1.html")

for i, html in enumerate(gen, 1):
    print(f"Page {i} length:", len(html))


Page 1 length: 50469
Page 2 length: 50877
Page 3 length: 51374
Page 4 length: 52524
Page 5 length: 51829
Page 6 length: 51602
Page 7 length: 51173
Page 8 length: 51144
Page 9 length: 50799
Page 10 length: 51145
Page 11 length: 50534
Page 12 length: 50600
Page 13 length: 51024
Page 14 length: 50778
Page 15 length: 51254
Page 16 length: 50812
Page 17 length: 50806
Page 18 length: 50872
Page 19 length: 51116
Page 20 length: 51109
Page 21 length: 50344
Page 22 length: 50674
Page 23 length: 51558
Page 24 length: 50672
Page 25 length: 51981
Page 26 length: 50512
Page 27 length: 51098
Page 28 length: 51441
Page 29 length: 50263
Page 30 length: 50600
Page 31 length: 50478
Page 32 length: 50631
Page 33 length: 51318
Page 34 length: 50846
Page 35 length: 50549
Page 36 length: 50701
Page 37 length: 50793
Page 38 length: 51639
Page 39 length: 50913
Page 40 length: 50516
Page 41 length: 50615
Page 42 length: 50726
Page 43 length: 50139
Page 44 length: 50142
Page 45 length: 50242
Page 46 length: 504

**2. Create a parse_items(html) generator that extracts all items from a single page and yields each item one-by-one instead of returning a list.**

In [2]:
import requests
from bs4 import BeautifulSoup

def parse_items(html):
    # Parse HTML 
    soup = BeautifulSoup(html, "html.parser")
    
    # Find all book containers
    books = soup.find_all("article", class_="product_pod")
    
    for book in books:
        # Extract book title
        title = book.h3.a["title"]
        
        # Extract book price
        price = book.find("p", class_="price_color").text
        
        # Extract book rating (e.g., "One", "Two", "Three", "Four", "Five")
        rating_tag = book.find("p", class_="star-rating")
        rating = rating_tag["class"][1] if rating_tag else None
        
        yield {
            "title": title,
            "price": price,
            "rating": rating
        }

# Example usage
url = "http://books.toscrape.com/catalogue/page-1.html"
html = requests.get(url).text
gen = parse_items(html)
print(next(gen))
for book in gen:
    print(book)


{'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three'}
{'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One'}
{'title': 'Soumission', 'price': 'Â£50.10', 'rating': 'One'}
{'title': 'Sharp Objects', 'price': 'Â£47.82', 'rating': 'Four'}
{'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'rating': 'Five'}
{'title': 'The Requiem Red', 'price': 'Â£22.65', 'rating': 'One'}
{'title': 'The Dirty Little Secrets of Getting Your Dream Job', 'price': 'Â£33.34', 'rating': 'Four'}
{'title': 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 'price': 'Â£17.93', 'rating': 'Three'}
{'title': 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 'price': 'Â£22.60', 'rating': 'Four'}
{'title': 'The Black Maria', 'price': 'Â£52.15', 'rating': 'One'}
{'title': 'Starving Hearts (Triangular Trade Trilogy, #1)', 'price': 'Â£13.99', 'rating': 'Two'}
{'title': "Shakespeare's S

**3. Build a scrape_all() generator that connects fetch_page() and parse_items() to yield scraped items lazily across all pages.**

In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- fetch_page generator ---
def fetch_page(url):
    """Yield HTML of each page until no next page exists."""
    while url:
        response = requests.get(url)
        html = response.text
        yield html
        
        soup = BeautifulSoup(html, "html.parser")
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            break
        url = urljoin(url, next_link.a["href"])

# --- parse_items generator ---
def parse_items(html):
    """Yield items (books) from a single page."""
    soup = BeautifulSoup(html, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    
    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating_tag = book.find("p", class_="star-rating")
        rating = rating_tag["class"][1] if rating_tag else None
        
        yield {
            "title": title,
            "price": price,
            "rating": rating
        }

# --- scrape_all generator ---
def scrape_all(start_url):
    """
    Lazily scrape all items from all pages.
    Yields one item at a time across all pages.
    """
    for html in fetch_page(start_url):      # fetch one page at a time
        for item in parse_items(html):      # parse one item at a time
            yield item

# --- Usage Example ---
start_url = "http://books.toscrape.com/catalogue/page-1.html"

for i, book in enumerate(scrape_all(start_url), 1):
    print(i, book)


1 {'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three'}
2 {'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One'}
3 {'title': 'Soumission', 'price': 'Â£50.10', 'rating': 'One'}
4 {'title': 'Sharp Objects', 'price': 'Â£47.82', 'rating': 'Four'}
5 {'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'rating': 'Five'}
6 {'title': 'The Requiem Red', 'price': 'Â£22.65', 'rating': 'One'}
7 {'title': 'The Dirty Little Secrets of Getting Your Dream Job', 'price': 'Â£33.34', 'rating': 'Four'}
8 {'title': 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 'price': 'Â£17.93', 'rating': 'Three'}
9 {'title': 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 'price': 'Â£22.60', 'rating': 'Four'}
10 {'title': 'The Black Maria', 'price': 'Â£52.15', 'rating': 'One'}
11 {'title': 'Starving Hearts (Triangular Trade Trilogy, #1)', 'price': 'Â£13.99', 'rating': 'Two'}
12

**4. Add lazy loading: ensure scrape_all() does not store any full page or full item list in memory.**

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- fetch_page generator ---
def fetch_page(url):
    """Yield HTML of each page until no next page exists."""
    while url:
        response = requests.get(url)
        yield response.text  # yield page HTML immediately
        
        soup = BeautifulSoup(response.text, "html.parser")
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            break
        url = urljoin(url, next_link.a["href"])

# --- parse_items generator ---
def parse_items(html):
    """Yield one item at a time from a single page."""
    soup = BeautifulSoup(html, "html.parser")
    for book in soup.find_all("article", class_="product_pod"):
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating_tag = book.find("p", class_="star-rating")
        rating = rating_tag["class"][1] if rating_tag else None
        yield {"title": title, "price": price, "rating": rating}

# --- scrape_all generator (fully lazy) ---
def scrape_all(start_url):
    """Yield items lazily across all pages, no full pages or lists stored."""
    for page_html in fetch_page(start_url):       # fetch one page at a time
        for item in parse_items(page_html):       # yield one item at a time
            yield item

# --- Example usage ---
start_url = "http://books.toscrape.com/catalogue/page-1.html"

for i, book in enumerate(scrape_all(start_url), 1):
    print(i, book)


1 {'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three'}
2 {'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One'}
3 {'title': 'Soumission', 'price': 'Â£50.10', 'rating': 'One'}
4 {'title': 'Sharp Objects', 'price': 'Â£47.82', 'rating': 'Four'}
5 {'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'rating': 'Five'}
6 {'title': 'The Requiem Red', 'price': 'Â£22.65', 'rating': 'One'}
7 {'title': 'The Dirty Little Secrets of Getting Your Dream Job', 'price': 'Â£33.34', 'rating': 'Four'}
8 {'title': 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 'price': 'Â£17.93', 'rating': 'Three'}
9 {'title': 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 'price': 'Â£22.60', 'rating': 'Four'}
10 {'title': 'The Black Maria', 'price': 'Â£52.15', 'rating': 'One'}
11 {'title': 'Starving Hearts (Triangular Trade Trilogy, #1)', 'price': 'Â£13.99', 'rating': 'Two'}
12

**5. Iterate over scrape_all() using a for loop and print the first 5 items to verify that your generator pipeline works.**

In [5]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- fetch_page generator ---
def fetch_page(url):
    """Yield HTML of each page until no next page exists."""
    while url:
        response = requests.get(url)
        yield response.text  # yield page HTML immediately
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            break
        url = urljoin(url, next_link.a["href"])

# --- parse_items generator ---
def parse_items(html):
    """Yield one item at a time from a single page."""
    soup = BeautifulSoup(html, "html.parser")
    for book in soup.find_all("article", class_="product_pod"):
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating_tag = book.find("p", class_="star-rating")
        rating = rating_tag["class"][1] if rating_tag else None
        yield {"title": title, "price": price, "rating": rating}

# --- scrape_all generator (fully lazy) ---
def scrape_all(start_url):
    """Yield items lazily across all pages, no full pages or lists stored."""
    for page_html in fetch_page(start_url):       # fetch one page at a time
        for item in parse_items(page_html):       # yield one item at a time
            yield item

# --- Example usage ---
start_url = "http://books.toscrape.com/catalogue/page-1.html"

for i, book in enumerate(scrape_all(start_url), 1):
    print(i, book)

    if i == 5:
        break


1 {'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three'}
2 {'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One'}
3 {'title': 'Soumission', 'price': 'Â£50.10', 'rating': 'One'}
4 {'title': 'Sharp Objects', 'price': 'Â£47.82', 'rating': 'Four'}
5 {'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'rating': 'Five'}


**6. Add a page counter inside fetch_page() to track how many pages were visited and print the total pages at the end.**

In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- fetch_page generator with page counter ---
def fetch_page(url):
    """Yield HTML of each page and count pages visited."""
    page_count = 0
    while url:
        page_count += 1
        print(f"Visiting page {page_count}: {url}")
        
        response = requests.get(url)
        yield response.text  # yield page HTML immediately
        
        soup = BeautifulSoup(response.text, "html.parser")
       # find <li class="next"> instead of <a> by text
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            break
        url = urljoin(url, next_link.a["href"])
    
    print(f"Total pages visited: {page_count}")

# --- Usage example ---
start_url = "http://books.toscrape.com/catalogue/page-1.html"

for page_html in fetch_page(start_url):
    pass  # just iterate to trigger page counter


Visiting page 1: http://books.toscrape.com/catalogue/page-1.html
Visiting page 2: http://books.toscrape.com/catalogue/page-2.html
Visiting page 3: http://books.toscrape.com/catalogue/page-3.html
Visiting page 4: http://books.toscrape.com/catalogue/page-4.html
Visiting page 5: http://books.toscrape.com/catalogue/page-5.html
Visiting page 6: http://books.toscrape.com/catalogue/page-6.html
Visiting page 7: http://books.toscrape.com/catalogue/page-7.html
Visiting page 8: http://books.toscrape.com/catalogue/page-8.html
Visiting page 9: http://books.toscrape.com/catalogue/page-9.html
Visiting page 10: http://books.toscrape.com/catalogue/page-10.html
Visiting page 11: http://books.toscrape.com/catalogue/page-11.html
Visiting page 12: http://books.toscrape.com/catalogue/page-12.html
Visiting page 13: http://books.toscrape.com/catalogue/page-13.html
Visiting page 14: http://books.toscrape.com/catalogue/page-14.html
Visiting page 15: http://books.toscrape.com/catalogue/page-15.html
Visiting page

**7. Validate generator behavior by confirming the script processes items only when needed (no preloading, no storing all data at once).**

In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- fetch_page generator with page counter ---
def fetch_page(url):
    """Yield HTML of each page lazily and track pages visited."""
    page_count = 0
    while url:
        page_count += 1
        print(f"[fetch_page] Fetching page {page_count}: {url}")
        
        response = requests.get(url)
        yield response.text  # Yield page HTML lazily
        
        soup = BeautifulSoup(response.text, "html.parser")
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            break
        url = urljoin(url, next_link.a["href"])
    
    print(f"Total pages visited: {page_count}")

# --- parse_items generator ---
def parse_items(html):
    """Yield one item at a time lazily from a page."""
    soup = BeautifulSoup(html, "html.parser")
    for book in soup.find_all("article", class_="product_pod"):
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating_tag = book.find("p", class_="star-rating")
        rating = rating_tag["class"][1] if rating_tag else None
        
        print(f"[parse_items] Processing book: {title}")  # Validate lazy processing
        yield {"title": title, "price": price, "rating": rating}

# --- scrape_all generator ---
def scrape_all(start_url):
    """Yield items lazily across all pages."""
    for page_html in fetch_page(start_url):
        for item in parse_items(page_html):
            yield item

# --- Test: validate lazy behavior ---
start_url = "http://books.toscrape.com/catalogue/page-1.html"

print("Start iterating over items...")

for i, book in enumerate(scrape_all(start_url), 1):
    print(f"[Main] Got item {i}: {book['title']}")
    if i == 5:  # Stop after 5 items for demonstration
        break

print("Done iterating.")


Start iterating over items...
[fetch_page] Fetching page 1: http://books.toscrape.com/catalogue/page-1.html
[parse_items] Processing book: A Light in the Attic
[Main] Got item 1: A Light in the Attic
[parse_items] Processing book: Tipping the Velvet
[Main] Got item 2: Tipping the Velvet
[parse_items] Processing book: Soumission
[Main] Got item 3: Soumission
[parse_items] Processing book: Sharp Objects
[Main] Got item 4: Sharp Objects
[parse_items] Processing book: Sapiens: A Brief History of Humankind
[Main] Got item 5: Sapiens: A Brief History of Humankind
Done iterating.


In [8]:
Create a fetch_page(url) generator that sends a GET request and yields the raw HTML of each page until no next page exists.
Create a parse_items(html) generator that extracts all items from a single page and yields each item one-by-one instead of returning a list.
Build a scrape_all() generator that connects fetch_page() and parse_items() to yield scraped items lazily across all pages.
Add lazy loading: ensure scrape_all() does not store any full page or full item list in memory.
Iterate over scrape_all() using a for loop and print the first 5 items to verify that your generator pipeline works.
Add a page counter inside fetch_page() to track how many pages were visited and print the total pages at the end.
Validate generator behavior by confirming the script processes items only when needed (no preloading, no storing all data at once).

SyntaxError: invalid syntax (3854836614.py, line 1)